# Análise Exploratória - Diabetes Mellitus no SUS

Este notebook apresenta uma análise exploratória dos dados processados pelo pipeline ETL.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuração
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

print('Bibliotecas importadas com sucesso!')

## 1. Carregamento dos Dados

In [ ]:
# Carregar dados processados
PROCESSED_DIR = Path('../data/processed')

parquet_file = PROCESSED_DIR / 'diabetes_sus_processed.parquet'
csv_file = PROCESSED_DIR / 'diabetes_sus_processed.csv'

if parquet_file.exists():
    df = pd.read_parquet(parquet_file)
    print(f'Dados carregados do Parquet: {len(df)} registros')
elif csv_file.exists():
    df = pd.read_csv(csv_file)
    print(f'Dados carregados do CSV: {len(df)} registros')
else:
    print('ERRO: Execute o pipeline primeiro!')
    print('Comando: python -m src.data.pipeline')

In [ ]:
# Visão geral
print(f'Registros: {len(df):,}')
print(f'Colunas: {len(df.columns)}')
print(f'\nColunas:')
for col in df.columns:
    print(f'  - {col}: {df[col].dtype}')

In [ ]:
# Primeiras linhas
df.head(10)

## 2. Estatísticas Descritivas

In [ ]:
# Estatísticas das colunas numéricas
colunas_numericas = ['internacoes', 'valor_total', 'dias_permanencia',
                     'obitos_hospitalares', 'obitos_sim', 'populacao',
                     'taxa_internacao_100k', 'taxa_mortalidade_100k']

colunas_existentes = [c for c in colunas_numericas if c in df.columns]
df[colunas_existentes].describe().round(2)

In [ ]:
# Valores ausentes
print('Valores ausentes por coluna:')
ausentes = df.isnull().sum()
ausentes[ausentes > 0]

## 3. Distribuição Temporal

In [ ]:
if 'ano' in df.columns and 'internacoes' in df.columns:
    internacoes_ano = df.groupby('ano')['internacoes'].sum()

    fig, ax = plt.subplots(figsize=(10, 5))
    internacoes_ano.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Internações por Diabetes Mellitus no SUS', fontsize=14)
    ax.set_xlabel('Ano')
    ax.set_ylabel('Número de Internações')
    plt.tight_layout()
    plt.show()

In [ ]:
if 'ano' in df.columns and 'obitos_sim' in df.columns:
    obitos_ano = df.groupby('ano')['obitos_sim'].sum()

    fig, ax = plt.subplots(figsize=(10, 5))
    obitos_ano.plot(kind='line', ax=ax, marker='o', color='crimson', linewidth=2)
    ax.set_title('Óbitos por Diabetes Mellitus (SIM)', fontsize=14)
    ax.set_xlabel('Ano')
    ax.set_ylabel('Número de Óbitos')
    plt.tight_layout()
    plt.show()

## 4. Distribuição por UF

In [ ]:
if 'uf' in df.columns and 'internacoes' in df.columns:
    internacoes_uf = df.groupby('uf')['internacoes'].sum().sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    internacoes_uf.plot(kind='barh', ax=ax, color='teal')
    ax.set_title('Internações por Diabetes Mellitus por UF', fontsize=14)
    ax.set_xlabel('Número de Internações')
    ax.set_ylabel('UF')
    plt.tight_layout()
    plt.show()

## 5. Distribuição por Sexo

In [ ]:
if 'sexo' in df.columns and 'internacoes' in df.columns:
    internacoes_sexo = df.groupby('sexo')['internacoes'].sum()

    fig, ax = plt.subplots(figsize=(8, 6))
    internacoes_sexo.plot(kind='pie', ax=ax, autopct='%1.1f%%',
                          colors=['#3498db', '#e74c3c', '#95a5a6'])
    ax.set_title('Internações por Sexo', fontsize=14)
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()

## 6. Distribuição por Faixa Etária

In [ ]:
if 'faixa_etaria' in df.columns and 'internacoes' in df.columns:
    internacoes_idade = df.groupby('faixa_etaria')['internacoes'].sum()

    # Ordenar
    ordem = ['0-4 anos', '5-9 anos', '10-14 anos', '15-19 anos',
             '20-29 anos', '30-39 anos', '40-49 anos', '50-59 anos',
             '60-69 anos', '70-79 anos', '80+ anos', 'Ignorado']
    ordem = [o for o in ordem if o in internacoes_idade.index]
    internacoes_idade = internacoes_idade.reindex(ordem)

    fig, ax = plt.subplots(figsize=(12, 5))
    internacoes_idade.plot(kind='bar', ax=ax, color='coral')
    ax.set_title('Internações por Faixa Etária', fontsize=14)
    ax.set_xlabel('Faixa Etária')
    ax.set_ylabel('Internações')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 7. Distribuição por CID-10

In [ ]:
if 'cid10' in df.columns and 'internacoes' in df.columns:
    internacoes_cid = df.groupby('cid10')['internacoes'].sum().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    internacoes_cid.plot(kind='bar', ax=ax, color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6'])
    ax.set_title('Internações por CID-10', fontsize=14)
    ax.set_xlabel('CID-10')
    ax.set_ylabel('Internações')
    plt.tight_layout()
    plt.show()

## 8. Custos Hospitalares

In [ ]:
if 'ano' in df.columns and 'valor_total' in df.columns:
    custos_ano = df.groupby('ano').agg({
        'valor_total': 'sum',
        'internacoes': 'sum',
    })
    custos_ano['valor_medio'] = custos_ano['valor_total'] / custos_ano['internacoes']

    fig, ax1 = plt.subplots(figsize=(10, 5))

    ax1.bar(custos_ano.index, custos_ano['valor_total'], color='steelblue', alpha=0.7)
    ax1.set_xlabel('Ano')
    ax1.set_ylabel('Valor Total (R$)', color='steelblue')

    ax2 = ax1.twinx()
    ax2.plot(custos_ano.index, custos_ano['valor_medio'], color='crimson', marker='o', linewidth=2)
    ax2.set_ylabel('Valor Médio (R$)', color='crimson')

    plt.title('Custos Hospitalares por Diabetes Mellitus', fontsize=14)
    plt.tight_layout()
    plt.show()

## 9. Indicadores por Região

In [ ]:
if 'regiao' in df.columns:
    ind_regiao = df.groupby('regiao').agg({
        'internacoes': 'sum',
        'obitos_sim': 'sum',
        'valor_total': 'sum',
        'populacao': 'sum',
    }).round(2)

    ind_regiao['taxa_internacao_100k'] = (ind_regiao['internacoes'] / ind_regiao['populacao'] * 100000).round(2)
    ind_regiao['taxa_mortalidade_100k'] = (ind_regiao['obitos_sim'] / ind_regiao['populacao'] * 100000).round(2)

    print('Indicadores por Região:')
    ind_regiao

---

**Fonte:** Dados públicos do DATASUS/SUS | **CID-10:** E10-E14 | **Aviso:** Dados agregados, sem identificadores individuais